In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/murtazaabdullah2010/apoai-task-1/train.json


In [4]:
import json

# with open("/bohr/training-set-h31o/v1/train.json", "r") as f:
#     train_ds = json.load(f)

with open("/kaggle/input/datasets/murtazaabdullah2010/apoai-task-1/train.json", "r") as f:
    train_ds = json.load(f)

train_ds = pd.DataFrame({
    "X" : train_ds.keys() , "y": train_ds.values()
})

In [5]:
train_ds

,X,y
0,Unternehmensgewinn,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
1,Brandruine,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 1]"
2,Tanningehalt,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]"
3,Warenkorb,"[0, 0, 0, 0, 1, 0, 0, 0, 1]"
4,Kaufhaus,"[0, 0, 0, 1, 0, 0, 0, 1]"
...,...,...
94301,Sprachtraining,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]"
94302,Sinneslust,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 1]"
94303,Vierzigstundenwoche,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, ..."
94304,Aufbauphase,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]"


In [6]:
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, resnet34, resnet50
from collections import Counter

In [7]:
def tokenize(word):
    return [w for w in word]

train_tokens = train_ds["X"].apply(tokenize)
def build_vocab(tokens):
    counter = Counter()
    for t in tokens:
        counter.update(t)
    vocab = {"<PAD>":0 ,"<UNK>":1}
    for w, c in counter.items():
        vocab[w] = len(vocab)
    return vocab

def build_seq(tokens, vocab):
    return [[vocab.get(t, 1) for t in w] for w in tokens]
vocab = build_vocab(train_tokens)
train_seq = build_seq(train_tokens, vocab)


In [8]:
# max_len = -1
# for idx, row in train_ds.iterrows():
#     max_len = max(max_len, len(row["X"]))

# max_len
max_len = 35

In [9]:
from torch.nn.utils.rnn import pad_sequence
class DF(Dataset):
    def __init__(self, seqs, targets):
        self.seqs = seqs
        self.targets = targets
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        return self.seqs[idx], (self.targets[idx])
def collate_fn(batch):
    x, y = zip(*batch)
    length = torch.tensor([len(seq) for seq in x] )
    x = [torch.tensor(seq, dtype=torch.long) for seq in x]
    y = [torch.tensor(tar, dtype=torch.long) for tar in y]
    x = pad_sequence(x,batch_first=True,padding_value=vocab["<PAD>"])
    y = pad_sequence(y,batch_first=True,padding_value=-100 )
    return x, y , length
train_df = DF(train_seq, (train_ds["y"]))
train_dl = DataLoader(train_df, batch_size = 16, shuffle = True, num_workers = 4, pin_memory = True, collate_fn = collate_fn)
for batch in train_dl:
    x, y , l= batch
    print(x.shape)
    print(y)
    print(l)
    break

torch.Size([16, 21])
tensor([[   0,    0,    0,    1,    0,    0,    0,    0,    1,    0,    0,    0,
            0,    0,    0,    1, -100, -100, -100, -100, -100],
        [   0,    0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    1, -100, -100, -100],
        [   0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    1, -100, -100, -100, -100, -100],
        [   0,    0,    0,    0,    1,    0,    0,    0,    0,    0,    1, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [   0,    0,    0,    1,    0,    0,    0,    1, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,
            0,    0,    0,    0,    0,    1, -100, -100, -100],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,
            0,    0,    0,  

In [14]:
# class MODEL(nn.Module):
#     def __init__(self, vocab_len):
#         super().__init__()
#         self.embeds = nn.Embedding(vocab_len, 256)
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=256,
#             nhead=4,
#             dropout=0.1,
#             batch_first=True
#         )
#         self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=4)
#         self.fc = nn.Sequential(
#             nn.Linear(256, 512),
#             nn.LayerNorm(512),
#             nn.GELU(),
#             nn.Dropout(0.1),
#             nn.Linear(512, 256),
#             nn.LayerNorm(256),
#             nn.GELU(),
#             nn.Dropout(0.05),
#             nn.Linear(256, 2)
#         )
#     def forward(self, x):
#         x = self.embeds(x)
#         out = self.encoder(x)
#         return self.fc(out)
# device = ("cuda" if torch.cuda.is_available() else "cpu")
# model = MODEL(len(vocab)).to(device)

In [35]:
class MODEL(nn.Module):
    def __init__(self, vocab_len):
        super().__init__()
        self.embeds= nn.Embedding(vocab_len, 128)
        self.lstm =nn.LSTM(128, 128, num_layers = 2, bidirectional = True, batch_first =True)
        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(256, 2)
        )
    def forward(self, x,l):
        x = self.embeds(x)
        x= torch.nn.utils.rnn.pack_padded_sequence(x, l.cpu(), batch_first = True, enforce_sorted= False)
        out, _ = self.lstm(x)
        out  , _= torch.nn.utils.rnn.pad_packed_sequence(out, batch_first= True)
        return self.fc(out)
device = ("cuda" if torch.cuda.is_available() else "cpu")
model = MODEL(len(vocab)).to(device)
model

MODEL(
  (embeds): Embedding(62, 128)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.05, inplace=False)
    (8): Linear(in_features=256, out_features=2, bias=True)
  )
)

In [36]:
total_zero= 0
total_one = 0
for idx, row in train_ds.iterrows():
    ys= (row["y"])
    for y in ys:
        if (y == 0):
            total_zero+= 1
        else:
            total_one += 1
print(total_one)
print(total_zero)

195233
1074134


In [37]:
one_w = total_one / (total_one + total_zero)
zero_w = total_zero / (total_one + total_zero)

In [40]:
model = MODEL(len(vocab)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=2e-5)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100, weight = torch.tensor([zero_w,one_w]).to(device))
for e in range(10):
    t_loss = 0
    model.train()
    for batch in train_dl:
        x, y, l = batch
        x, y, l = x.to(device), y.to(device), l.to(device)
        out = model(x, l)  
        loss = loss_fn(out.reshape(-1, out.size(-1)),  y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        t_loss += loss.item()
    torch.cuda.empty_cache()
    print(f"E: {e} train_loss: {t_loss/len(train_dl)}")

E: 0 train_loss: 0.22861641586787423
E: 1 train_loss: 0.12251315284536186
E: 2 train_loss: 0.09748146176350835
E: 3 train_loss: 0.08407177120665178
E: 4 train_loss: 0.07520624496899007
E: 5 train_loss: 0.06851038376471316
E: 6 train_loss: 0.0633602874620523
E: 7 train_loss: 0.05901142152423628
E: 8 train_loss: 0.05522431765123728


Exception in thread Thread-27 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7be704e22e80>
Trace

KeyboardInterrupt: 

In [ ]:
import json
import torch
import torch.nn.functional as F
import os
import zipfile
import logging
def predict_and_save(model, input_file, output_file, device="cpu"):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    model.eval()
    predictions = {}
    with torch.no_grad():
        for word, _ in data.items():
            idxs = [vocab.get(c, 1) for c in word]
            x = torch.tensor(idxs).unsqueeze(0).to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=-1)
            boundary_probs = probs[0, :, 1].cpu().numpy()
            boundaries = (boundary_probs > 0.5).astype(int)
            boundaries = boundaries.tolist()[:len(word)]
            predictions[word] = boundaries     
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=4, ensure_ascii=False)
    logging.info(f"Predictions saved to {output_file}")

In [ ]:
## Obtain the validation set and test set for AB list evaluation
## 【Only available after the notebook is submitted to the competition】
## 【It is normal to encounter errors during the code debugging phase 
##  because the following data is not disclosed to the participants 
## and cannot be read at this stage】

if os.environ.get('ANSWER_PATH'):
    PATH = os.environ.get("ANSWER_PATH") + "/" 
else:
    print("When the Baseline is running, this error occurs because it cannot read the test set, which is a normal phenomenon.")  
    
# Predict and save results
input_file = PATH + "val.json"
output_file = "submissionval.json"
predict_and_save(model, input_file, output_file, device)

# Predict and save results
input_file = PATH + "test.json"
output_file = "submissiontest.json"
predict_and_save(model, input_file, output_file, device)

In [ ]:
# Define the files to be packaged and the compressed file name.
files_to_zip = ['submissionval.json', 'submissiontest.json']
zip_filename = 'submission.zip'

# Create a zip file.
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add files to the zip file.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')